In [ ]:
import bz2
import json
import pandas as pd

from collections import Counter
from itertools import islice
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "wikidata"
WIKIDATA_PATH = RAW_DIR / "latest-all.json.bz2"

assert WIKIDATA_PATH.exists()

SAMPLE_SIZE = 1000

def iter_wikidata_entities(path: Path):
    with bz2.open(path, "rt", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line in ("[", "]"):
                continue
            if line.endswith(","):
                line = line[:-1]
            yield json.loads(line)

def sample_entities(path: Path, limit: int = SAMPLE_SIZE):
    return list(islice(iter_wikidata_entities(path), limit))

def summarize_entity(entity: dict) -> dict:
    return {
        "id": entity.get("id"),
        "type": entity.get("type"),
        "modified": entity.get("modified"),
        "lastrevid": entity.get("lastrevid"),
        "label_langs": len(entity.get("labels", {})),
        "alias_langs": len(entity.get("aliases", {})),
        "description_langs": len(entity.get("descriptions", {})),
        "claim_properties": len(entity.get("claims", {})),
        "sitelinks": len(entity.get("sitelinks", {})),
    }

def flatten_labels(entities: list[dict]) -> pd.DataFrame:
    rows = []
    for entity in entities:
        for lang, value in entity.get("labels", {}).items():
            rows.append(
                {
                    "id": entity.get("id"),
                    "entity_type": entity.get("type"),
                    "lang": lang,
                    "label": value.get("value"),
                }
            )
    return pd.DataFrame(rows)

def flatten_aliases(entities: list[dict]) -> pd.DataFrame:
    rows = []
    for entity in entities:
        for lang, aliases in entity.get("aliases", {}).items():
            for alias in aliases:
                rows.append(
                    {
                        "id": entity.get("id"),
                        "entity_type": entity.get("type"),
                        "lang": lang,
                        "alias": alias.get("value"),
                    }
                )
    return pd.DataFrame(rows)

def flatten_claim_counts(entities: list[dict]) -> pd.DataFrame:
    counter = Counter()
    for entity in entities:
        for prop, claims in entity.get("claims", {}).items():
            counter[prop] += len(claims)
    return (
        pd.DataFrame(
            [{"property": prop, "claims_in_sample": count} for prop, count in counter.items()]
        )
        .sort_values("claims_in_sample", ascending=False)
        .reset_index(drop=True)
    )

In [ ]:
entities = sample_entities(WIKIDATA_PATH, SAMPLE_SIZE)

print(f"Sampled entities: {len(entities)}")

In [ ]:
df_entities = pd.DataFrame(summarize_entity(entity) for entity in entities)
df_entities.head()

In [ ]:
df_labels = flatten_labels(entities)
df_labels.head()

In [ ]:
df_aliases = flatten_aliases(entities)
df_aliases.head()

In [ ]:
df_claim_counts = flatten_claim_counts(entities)

pd.DataFrame(
    {
        "entity_type": df_entities["type"].value_counts().index,
        "count": df_entities["type"].value_counts().values,
    }
)

In [ ]:
top_label_langs = (
    df_labels["lang"].value_counts().rename_axis("lang").reset_index(name="label_count")
)

top_alias_langs = (
    df_aliases["lang"].value_counts().rename_axis("lang").reset_index(name="alias_count")
    if not df_aliases.empty
    else pd.DataFrame(columns=["lang", "alias_count"])
)

print("Top label languages")
display(top_label_langs.head(20))

print("\nTop alias languages")
display(top_alias_langs.head(20))

print("\nTop claim properties in sample")
display(df_claim_counts.head(20))